# Transformation du problème d'équilibre de Nash en problème d'optimisation linéaire mixte (MILP)

Ce document décrit comment modéliser la recherche d'un équilibre de Nash pour un jeu à deux joueurs sous la forme d'un problème MILP. Nous présentons ici les contraintes principales qui lient les stratégies, les gains purs, le support (via des variables binaires) et les regrets, ainsi que la formulation de l'objectif.


## 1.1 Définition du jeu

Un jeu est défini par deux matrices de gains \\( A \\) et \\( B \\) :  
- Pour le joueur 1 (Ligne), la matrice \\( A \\) décrit les gains.  
- Pour le joueur 2 (Colonne), la matrice \\( B \\) décrit les gains.  

Dans le cas particulier d'un jeu à somme nulle, on a :  
$$
B = -A
$$

Une **stratégie mixte** pour un joueur est une distribution de probabilités sur ses stratégies pures.  

Pour le joueur 1 (ligne) :  
$$
X = \left\{ x \in \mathbb{R}^m \;\middle|\; x_i \ge 0,\; \sum_{i=1}^{m} x_i = 1 \right\}
$$

Pour le joueur 2 (colonne) :  
$$
Y = \left\{ y \in \mathbb{R}^n \;\middle|\; y_j \ge 0,\; \sum_{j=1}^{n} y_j = 1 \right\}
$$

Un couple \( (x, y) \) est un **équilibre de Nash** si et seulement si :

- \( x \) est la meilleure réponse à \( y \), c’est-à-dire :
$$
x^T A y \ge z^T A y \quad \forall z \in X
$$

- \( y \) est la meilleure réponse à \( x \), c’est-à-dire :
$$
x^T B y \ge x^T B w \quad \forall w \in Y
$$

Dans le cas à somme nulle (\( B = -A \)), cela devient :
$$
x^T A y \le x^T A w
$$

On note la **valeur du jeu** par :
$$
v = x^T A y
$$

---

## 1.2 Fondements Théoriques

### 1.2.1 Stratégies mixtes et gains attendus

- **Stratégie mixte de Ligne :**  
$$
x \in X = \left\{ x \in \mathbb{R}^m \;\middle|\; x_i \ge 0,\; \sum_{i=1}^{m} x_i = 1 \right\}
$$

- **Stratégie mixte de Colonne :**  
$$
y \in Y = \left\{ y \in \mathbb{R}^n \;\middle|\; y_j \ge 0,\; \sum_{j=1}^{n} y_j = 1 \right\}
$$

- **Gain attendu :**  
Si Ligne joue \( x \) et Colonne joue \( y \), alors :

- Gain de Ligne :
$$
x^T A y
$$

- Gain de Colonne :
$$
x^T B y
$$


In [1]:
import numpy as np

def est_equilibre_nash(A, B, x, y):
    # Gains attendus des joueurs
    gain_x = np.dot(x, np.dot(A, y))  # Gain du joueur Ligne
    gain_y = np.dot(x, np.dot(B, y))  # Gain du joueur Colonne
    
    # Vérification pour le joueur Ligne
    for i in range(A.shape[0]):
        z = np.zeros_like(x)
        z[i] = 1  # Stratégie pure
        if np.dot(z, np.dot(A, y)) > gain_x:
            return False 
    
    # Vérification pour le joueur Colonne
    for j in range(B.shape[1]):
        w = np.zeros_like(y)
        w[j] = 1  # Stratégie pure
        if np.dot(x, np.dot(B, w)) > gain_y:
            return False 

    return True  # (x, y) est un équilibre de Nash


In [2]:
# Exemple :
A = np.array([[3, -1], [0, 2]])
B = -A
x = np.array([0.5, 0.5])
y = np.array([0.4, 0.6])

print("Équilibre de Nash :", est_equilibre_nash(A, B, x, y))

Équilibre de Nash : False


### 1.2.2 Condition d'indifférence et regrets

### Pourquoi tout doit atteindre le gain maximal ?

L'idée fondamentale pour obtenir un **équilibre de Nash** est la suivante :

---

####  Condition d'indifférence

Pour qu'un joueur n’ait **aucun intérêt à changer unilatéralement de stratégie**, toutes les actions **pures utilisées** par ce joueur (c’est-à-dire, celles pour lesquelles \( $x_i$ > 0 \)) doivent rapporter **exactement le gain maximal**.

---

#### 🔹 Pour le joueur 1 (Ligne)

Si \( $x_i$ > 0 \), alors on doit avoir :

$$
e_i^T A y = M_x
$$

où

$$
M_x = \max_{k=1,\dots,m} \, (e_k^T A y)
$$

est le **gain maximal** obtenu par une stratégie pure.

Le **regret** pour l’action \( i \) est défini par :

$$
r_{x,i} = M_x - e_i^T A y
$$

Donc, la condition d’indifférence implique :

$$
x_i > 0 \Rightarrow r_{x,i} = 0
$$

---

#### 🔹 Pour le joueur 2 (Colonne)

Si \( $y_j$ > 0 \), alors :

$$
x^T B e_j = M_y
$$

où

$$
M_y = \max_{l=1,\dots,n} \, (x^T B e_l)
$$

et le **regret** de l’action \( j \) pour Colonne est :

$$
r_{y,j} = M_y - x^T B e_j
$$

Donc :

$$
y_j > 0 \Rightarrow r_{y,j} = 0
$$

---

###  En résumé

Chaque action **utilisée** par un joueur dans une stratégie mixte doit rapporter **le gain maximal possible** :

- $e_i^T A y = M_x$ pour le joueur 1  
- $x^T B e_j = M_y$ pour le joueur 2

Sinon, le joueur pourrait **réallouer de la probabilité** vers une action plus rentable, **brisant l’équilibre**.

 **C’est cette égalité des gains sur le support qui définit l’indifférence**, cœur de l’équilibre de Nash.




## 2. Modélisation MILP du problème

Pour transformer ce problème en un problème d'optimisation, nous utilisons plusieurs types de variables et imposons des contraintes qui forcent les conditions d'indifférence et d'optimalité.

### 2.1 Variables

1. **Variables continues pour les stratégies**  
   - $x_i$ pour $i = 1, \dots, m$ (pour le joueur 1).
   - $y_j$ pour $j = 1, \dots, n$ (pour le joueur 2).

2. **Variables binaires pour le support**  
   - $b_x[i]$ pour indiquer si $x_i > 0$.
   - $b_y[j]$ pour indiquer si $y_j > 0$.

3. **Variables continues pour les gains maximaux**  
   - $M_x$ représente le gain maximal pur attendu pour le joueur 1, c'est-à-dire le maximum des gains obtenus en jouant une stratégie pure contre $y$.
   - $M_y$ représente le gain maximal pour le joueur 2.

4. **Variables de regret**  
   Pour chaque action pure $i$ du joueur 1, nous définissons le regret :  
   $$
   r_{x,i} = M_x - e_i^T A y.
   $$
   De même, on peut définir pour le joueur 2,  
   $$
   r_{y,j} = M_y - x^T B e_j.
   $$
   (La minimisation des regrets permet de forcer l'égalité entre le gain pur d'une action utilisée et le gain maximal.)

### 2.2 Contraintes

Les contraintes de notre modèle se répartissent en plusieurs catégories :

#### 2.2.1 Contraintes de probabilité

Pour garantir que $x$ et $y$ sont bien des distributions de probabilités, nous imposons :

$$
\sum_{i=1}^{m} x_i = 1 \quad \text{et} \quad x_i \ge 0 \quad \forall i,
$$

$$
\sum_{j=1}^{n} y_j = 1 \quad \text{et} \quad y_j \ge 0 \quad \forall j.
$$

#### 2.2.2 Lien entre les variables continues et les variables binaires

Pour forcer que $b_x[i] = 1$ si $x_i > 0$ (et de même pour $b_y[j]$), nous imposons :

$$
x_i \le b_x[i] \quad \text{et} \quad x_i \ge \epsilon\, b_x[i] \quad \forall i,
$$

$$
y_j \le b_y[j] \quad \text{et} \quad y_j \ge \epsilon\, b_y[j] \quad \forall j,
$$

où $\epsilon > 0$ est un petit réel.

#### 2.2.3 Contraintes sur le gain pur et la condition d'indifférence

Pour chaque action pure $i$ du joueur 1, le gain pur est défini par :

$$
g_i = e_i^T A y.
$$

On impose :

$$
g_i \le M_x \quad \forall i.
$$

Pour forcer que, si $x_i > 0$ (c'est-à-dire si $b_x[i] = 1$), alors $g_i$ doit être égal à $M_x$, nous utilisons une contrainte de type big‑M.  
Au lieu d'une constante arbitraire, nous utilisons le maximum des valeurs absolues de $A$ :

$$
M_{\text{max}} = \max_{i,j} |A_{ij}|.
$$

La contrainte devient alors :

$$
g_i \ge M_x - M_{\text{max}} (1-b_x[i]) \quad \forall i.
$$

De plus, nous définissons le regret pour l'action $i$ du joueur 1 par :

$$
r_{x,i} \ge M_x - e_i^T A y \quad \forall i.
$$

Pour le joueur 2, on définit de façon analogue, pour chaque action pure $j$ :

$$
h_j = x^T B e_j,
$$

avec les contraintes :

$$
h_j \le M_y \quad \forall j,
$$

$$
h_j \ge M_y - M'_{\text{max}} (1-b_y[j]) \quad \forall j,
$$

où $M'_{\text{max}} = \max_{i,j} |B_{ij}|$.

Et le regret pour le joueur 2 est défini par :

$$
r_{y,j} \ge M_y - x^T B e_j \quad \forall j.
$$

#### 2.2.4 Fonction objectif

L'objectif peut être formulé de manière à maximiser la valeur globale du jeu tout en forçant l'égalité des gains dans le support (c'est-à-dire, en minimisant les regrets). Par exemple :

$$
\max \left( M_x + M_y - \lambda \left(\sum_{i=1}^{m} r_{x,i} + \sum_{j=1}^{n} r_{y,j}\right) \right),
$$

où $\lambda > 0$ est un coefficient de pondération.  
Si l'échelle des regrets et des gains est compatible, on peut choisir $\lambda = 1$.


### 2.3 Implémentation en Python

L'implémentation suivante modélise le problème MILP en utilisant la bibliothèque `pulp` :

In [6]:
import pulp as pl

def probleme(A,B):
    
    m, n = A.shape

    prob = pl.LpProblem("Nash", pl.LpMaximize)

    # Variables de stratégie pour Ligne et Colonne
    x = [pl.LpVariable(f"x_{i}", lowBound=0) for i in range(m)]
    y = [pl.LpVariable(f"y_{j}", lowBound=0) for j in range(n)]

    # Variables binaires pour le support de x et de y
    b_x = [pl.LpVariable(f"b_x_{i}", cat="Binary") for i in range(m)]
    b_y = [pl.LpVariable(f"b_y_{i}", cat="Binary") for i in range(n)]

    # Valeur maximale attendue pour Ligne et pour Colonne
    M_x = pl.LpVariable("M_x", lowBound=None)
    M_y = pl.LpVariable("M_y", lowBound=None)

    # Variables de regret pour chaque action de Ligne et Colonne
    r_x = [pl.LpVariable(f"r_x_{i}", lowBound=0) for i in range(m)]
    r_y = [pl.LpVariable(f"r_y_{i}", lowBound=0) for i in range(n)]

    # Contraintes de probabilité
    prob += pl.lpSum(x) == 1, "Sum_x"
    prob += pl.lpSum(y) == 1, "Sum_y"

    # Contraintes de lien entre x (respectivement y)
    # et le support lié b_x (respectivement b_y)
    for i in range(m):
        prob += x[i] <= b_x[i], f"Link_x_b_{i}"
        #prob += x[i] >= epsilon * b_x[i], f"LowerBound_x_{i}"
    for i in range(n):
        prob += y[i] <= b_y[i], f"Link_y_b_{i}"
        #prob += y[i] >= epsilon * b_y[i], f"LowerBound_y_{i}"

    for i in range(m):
        Ay_i = pl.lpSum([A[i, j] * y[j] for j in range(n)])
        # On impose que Ay_i <= M_x pour tout i
        prob += Ay_i <= M_x, f"UpperBound_Ay_{i}"
        # Si l'action est utilisée (b_x[i] = 1), on impose que Ay_i = M_x <=> ri = 0
        # sinon ri <= M_x - max A_i_j
        prob += Ay_i >= M_x - np.max(A)*(1 - b_x[i]), f"Equality_Ay_{i}"
        # Définir le regret pour l'action i : r_x_i = M_x - Ay_i
        prob += r_x[i] >= M_x - Ay_i, f"Regret_x_{i}"

    for i in range(n):
        xB_i = pl.lpSum([x[j] * B[j, i] for j in range(m)])
        # On impose que Ay_i <= M_y pour tout i
        prob += xB_i <= M_y, f"UpperBound_xB_{i}"
        # Si l'action est utilisée (b_y[i] = 1), on impose que xB_i = M_y <=> ri = 0
        # sinon ri <= M_y - max B_i_j
        prob += xB_i >= M_y - np.max(B)*(1 - b_y[i]), f"Equality_xB_{i}"
        # Définir le regret pour l'action i : r_y_i = M_x - xB_i
        prob += r_y[i] >= M_y - xB_i, f"Regret_y_{i}"

    # Objectif : maximiser la valeur du jeu pour Ligne
    prob += M_x + M_y - pl.lpSum(r_x) - pl.lpSum(r_y), "Maximize_Game_Value"

    prob.solve()
    
    # Affichage des résultats
    print("Status:", pl.LpStatus[prob.status])
    print("Optimal value of M_x =", pl.value(M_x))
    print("Optimal value of M_y =", pl.value(M_y))
    print("Strategy for Ligne (x):")
    for i in range(m):
        print(f"  x[{i}] =", pl.value(x[i]), "with binary b_x =", pl.value(b_x[i]))
    print("Strategy for Colonne (y):")
    for j in range(n):
        print(f"  y[{j}] =", pl.value(y[j]), "with binary b_y =", pl.value(b_y[j]))
    


## 3. Récapitulatif du modèle MILP

Le modèle MILP pour l'équilibre de Nash se compose donc de :

1. **Variables :**
   - $x_i$ et $y_j$ (stratégies mixtes)
   - $b_x[i]$ et $b_y[j]$ (indicateurs de support)
   - $M_x$ et $M_y$ (gains maximaux)
   - $r_{x,i}$ et $r_{y,j}$ (regrets)

2. **Contraintes :**
   - **Probabilité :** $\sum_{i=1}^{m} x_i = 1$, $x_i \ge 0$, $\sum_{j=1}^{n} y_j = 1$, $y_j \ge 0$.
   - **Lien entre continus et binaires :** $x_i \le b_x[i]$, $x_i \ge \epsilon\, b_x[i]$, etc.
   - **Gain pur et condition d'indifférence pour Ligne :**  
     $g_i = e_i^T A y \le M_x$ et  
     $g_i \ge M_x - M_{\text{max}} (1-b_x[i])$ pour tout $i$.  
     De plus, $r_{x,i} \ge M_x - e_i^T A y$.
   - **Contraintes analogues pour Colonne :**  
     $h_j = x^T B e_j \le M_y$ et  
     $h_j \ge M_y - M'_{\text{max}} (1-b_y[j])$ pour tout $j$.  
     De plus, $r_{y,j} \ge M_y - x^T B e_j$.

3. **Objectif :**  
   $$
   \max \left( M_x + M_y - \lambda \left(\sum_{i=1}^{m} r_{x,i} + \sum_{j=1}^{n} r_{y,j}\right) \right).
   $$

Cette formulation permet d'obtenir une solution où, pour chaque action utilisée par un joueur, le gain pur est égal au gain maximal attendu (c'est-à-dire, le regret est nul), assurant ainsi l'indifférence nécessaire à un équilibre de Nash.



In [10]:
A = np.array([[3, 1],
              [2, 4]])
B = np.array([[1, 2],
              [3, 0]])
probleme(A,B)

Status: Optimal
Optimal value of M_x = 2.5
Optimal value of M_y = 1.5
Strategy for Ligne (x):
  x[0] = 0.75 with binary b_x = 1.0
  x[1] = 0.25 with binary b_x = 1.0
Strategy for Colonne (y):
  y[0] = 0.75 with binary b_y = 1.0
  y[1] = 0.25 with binary b_y = 1.0


In [11]:
# Cas d'un jeu à somme nulle simple
A = np.array([[3, 0],
              [0, 3],
              [1, 1]])
B = np.array([[-3, 0],
              [0, -3],
              [-1, -1]])
probleme(A,B)

Status: Optimal
Optimal value of M_x = 1.5
Optimal value of M_y = -1.5
Strategy for Ligne (x):
  x[0] = 0.5 with binary b_x = 1.0
  x[1] = 0.5 with binary b_x = 1.0
  x[2] = 0.0 with binary b_x = 0.0
Strategy for Colonne (y):
  y[0] = 0.5 with binary b_y = 1.0
  y[1] = 0.5 with binary b_y = 1.0


In [13]:

#dilemme du prisonnier

A = np.array([[3, 0],
              [5, 1]])
B = np.array([[3, 5],
              [0, 1]])

probleme(A,B)

Status: Optimal
Optimal value of M_x = 1.0
Optimal value of M_y = 1.0
Strategy for Ligne (x):
  x[0] = 0.0 with binary b_x = 0.0
  x[1] = 1.0 with binary b_x = 1.0
Strategy for Colonne (y):
  y[0] = 0.0 with binary b_y = 0.0
  y[1] = 1.0 with binary b_y = 1.0


In [17]:
# Définition du jeu Pierre-Feuille-Ciseaux (PFC)
# Gains pour le joueur 1
A = np.array([
    [ 0, -1,  1],
    [ 1,  0, -1],
    [-1,  1,  0]
])

# Jeu à somme nulle => B = -A
B = -A

# Appel du solveur MILP
probleme(A, B)

Status: Optimal
Optimal value of M_x = 0.0
Optimal value of M_y = 0.0
Strategy for Ligne (x):
  x[0] = 0.33333333 with binary b_x = 1.0
  x[1] = 0.33333333 with binary b_x = 1.0
  x[2] = 0.33333333 with binary b_x = 1.0
Strategy for Colonne (y):
  y[0] = 0.33333333 with binary b_y = 1.0
  y[1] = 0.33333333 with binary b_y = 1.0
  y[2] = 0.33333333 with binary b_y = 1.0


## Exemple d'application du modèle MILP à un grand jeu aléatoire

In [19]:
import numpy as np

def generate_random_game(m=10, n=10, zero_sum=False, seed=None):
    rng = np.random.default_rng(seed)
    A = rng.integers(low=-100, high=100, size=(m, n))

    if zero_sum:
        B = -A
    else:
        B = rng.integers(low=-100, high=100, size=(m, n))

    return A, B

In [21]:
# Exemple random
A, B = generate_random_game(m=25, n=25, seed=42)
print("Matrice A (Joueur 1):\n", A)
print("\nMatrice B (Joueur 2):\n", B)

# Appel du solveur MILP
probleme(A, B)

Matrice A (Joueur 1):
 [[-83  54  30 -13 -14  71 -83  39 -60 -82   5  95  47  52  43  57   2 -75
   67 -10   0 -26 -64  85  56]
 [ 28 -20  64   9 -12 -10 -55 -82  10  77 -88  71  65 -45  26 -67  51  40
  -30 -87  94 -11  78  35  55]
 [ 51 -62 -28  -7  -1 -92   9 -70  48  36  84  48 -27  93 -18 -35  81 -26
  -85  -7  59 -63  -8 -75  37]
 [ -5 -34 -55  12  33  88 -13 -68  66  25  40 -81 -38  53  66 -13  60  68
  -23  79 -43 -53  36  27 -73]
 [ 66 -61  60 -99  59  57  56  32  -6  41 -45  56  11  -9   1  13 -93 -73
  -51 -78 -13  33  30  -6  71]
 [ 13 -85  52  14  26  13  10 -82  11  58 -40  20 -94 -31 -13  96 -58 -45
  -19  98  70 -94 -54  64 -89]
 [ 71 -44  83 -42 -14  32 -75  11   1  56  99  32 -19 -19 -17  62 -36 -67
  -34 -96 -79 -82  54  44  39]
 [ -8  43 -68  80   0  87 -70  -1  39  -1 -11 -67 -24 -53 -40  36  26  21
  -28  91 -83 -32 -77 -33  92]
 [-27  81  -1  39  -9 -47  52  93 -48  55 -48  43  57 -11  47 -46 -85 -81
  -11  80 -75  -9  40 -60  44]
 [-39  61  15   9 -65  -8  71 -9